# Chapter 17: Wiring the Airlock (Reference)

## Learning Objectives

- Compose Gate 1, Gate 2, and Gate 3 into one Decision with decide()
- Attempt native auto-merge enrollment conditioned on that Decision
- Watch a failing gate correctly block enrollment and name why
- Explain why the real workflows need none of this composition code

## Setup

The next cell sets up reproducibility and the `PRA_MODE` toggle. You should see `PRA_MODE = 'fixture'` printed by default.

In [ ]:
import os
import random
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pr_automerge").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

RANDOM_STATE: int = 42
random.seed(RANDOM_STATE)

PRA_MODE = os.environ.get("PRA_MODE", "fixture")
PRA_REPO = os.environ.get("PRA_REPO", "")

if PRA_MODE == "live":
    assert PRA_REPO, "Set PRA_REPO=owner/name to run against a real repo"

print(f"PRA_MODE = {PRA_MODE!r}")


## 1. A PR That Passes All Three Gates

The next cell builds a sample `PRMetadata` and calls `decide()` with every input healthy. You should see all three gates PASS and a final `MERGE` verdict.

In [ ]:
from pr_automerge.models import PRMetadata
from pr_automerge.render import gate_table, decision_line
from labs.lab_17_airlock import decide, enqueue_for_automerge

pr = PRMetadata(
    number=102, title="feat: add greeting helper to sandbox app",
    base="main", head="feat/greeting-helper",
    additions=100, deletions=20, changed_files=6, critical_path_hits=0,
)
decision = decide(
    pr, main_exists=True, protection_configured=True,
    required_checks_registered=True, auto_merge_enabled=True,
    ci_conclusion="success",
)
gate_table(decision.gates)
decision_line(decision)


## 2. Attempt Enrollment

The next cell calls `enqueue_for_automerge` with the passing decision above. You should see `enrolled=True`.

In [ ]:
result = enqueue_for_automerge("example/example", pr.number, decision)
print(f"attempted={result['attempted']} enrolled={result['enrolled']}")
print(f"  {result['detail']}")


## 3. The Same PR, But CI Is Red

The next cell re-decides the same PR with `ci_conclusion="failure"`. You should see Gate 2 FAIL, a `HOLD` verdict, and enrollment correctly refused without even attempting the `gh pr merge` call.

In [ ]:
decision2 = decide(
    pr, main_exists=True, protection_configured=True,
    required_checks_registered=True, auto_merge_enabled=True,
    ci_conclusion="failure",
)
gate_table(decision2.gates)
decision_line(decision2)

result2 = enqueue_for_automerge("example/example", pr.number, decision2)
print(f"\nattempted={result2['attempted']} enrolled={result2['enrolled']}")
print(f"  {result2['detail']}")


## Takeaways & Next Steps

This notebook's takeaway is that `decision.merge` alone decided whether enrollment was even attempted -- exactly the property that makes this composition safe to reason about.

In [ ]:
print("Re-run this notebook with PRA_MODE=live to see it against the real sandbox repo.")


---

📖 **Reading companion:** [Chapter 17: Wiring the Airlock](../learning_modules/chapter_17_wiring_the_airlock.md)
